# Sonuç sunumu

**Bu notebook İSTATİSTİK HESAPLAMAZ.** Bütün hesap `analysis/analyze.py` içinde yapılır ve
makine-okunur dosyalara yazılır; burada yalnız o dosyalar okunup tablolaştırılır.

Gerekçe: notebook hücreleri sırasız/kısmi çalıştırılabilir ve çıktı ile kod kolayca
birbirinden kopar. Ön-kayıtlı estimand'ın notebook'ta yeniden hesaplanması,
bildiride raporlanan sayının hangi kodla üretildiğini belirsizleştirirdi.

Önce analizi çalıştır:

```
uv run python -m analysis.analyze --exp logs/exp_<name>
```

Ek bağımlılık yok (yalnız standart kütüphane). Şekiller P4'te ayrı eklenecek.

In [ ]:
import csv
import json
from pathlib import Path

EXP = Path("../logs/exp_gemini_main")  # <-- analiz edilecek deney
OUT = EXP / "analysis"

summary = json.loads((OUT / "analysis_summary.json").read_text(encoding="utf-8"))
effects = json.loads((OUT / "paired_effects.json").read_text(encoding="utf-8"))
task_level = list(csv.DictReader((OUT / "task_level.csv").open(encoding="utf-8")))

print(f"{summary['experiment']} | {summary['model']} | {summary['task_set']}")
print(f"{summary['n_tasks']} görev × {summary['repeats']} tekrar × {len(summary['arms'])} kol")
print(f"analiz birimi: {summary['analysis_unit']} | birincil metrik: {summary['primary_metric']}")
if summary["preliminary"]:
    print("\nUYARI: PRELIMINARY — bildiriye girmez.")
    print(f"  eksik koşu: {len(summary['integrity']['missing'])}")
    print(f"  kullanım verisi eksik koşu: "
          f"{len(summary['call_log_check']['runs_without_usage'])}")

## Tablo 1 — Kol başına görev-düzeyi ortalama geçme oranı

In [ ]:
print(f"{'kol':28s} {'Plus':>8s} {'Base':>8s}   gözlem")
for arm in summary["arms"]:
    s = summary["arm_summary"][arm]
    print(f"{arm:28s} {s['plus_pass']['task_mean_rate']:8.3f} "
          f"{s['base_pass']['task_mean_rate']:8.3f}   "
          f"{s['plus_pass']['observation_count']}")

## Tablo 2 — Eşleştirilmiş etkiler (%95 cluster-bootstrap CI)

Örnekleme birimi görevdir. p-değeri bilinçli olarak raporlanmaz (ön-kayıt, §8).

In [ ]:
for metric in ("plus_pass", "base_pass"):
    print(f"\n[{metric}]")
    for name, e in effects[metric].items():
        print(f"  {name:38s} {e['point_estimate']:+.3f}  "
              f"[{e['ci_low']:+.3f}, {e['ci_high']:+.3f}]  "
              f"lehte {e['tasks_favoring_a']} / aleyhte {e['tasks_favoring_b']} / "
              f"berabere {e['tasks_tied']}  (n={e['n_tasks']} görev)")

## Tablo 3 — Maliyet ve kullanım (ölçüm üretenler vs. altyapı overhead'i)

In [ ]:
u = summary["usage"]


def f(value, spec):
    """Eksik kullanım verisi None gelir — 0 gibi biçimlendirilmemeli."""
    return format(value, spec) if value is not None else "—".rjust(len(format(0, spec)))


if not u["complete"]:
    print("UYARI: kullanım verisi eksik — maliyet/token alanları null (sıfır DEĞİL).\n")

print(f"{'kol':28s} {'koşu':>6s} {'çağrı':>7s} {'girdi tok':>10s} {'çıktı tok':>10s} "
      f"{'maliyet $':>10s} {'medyan $/koşu':>14s}")
for arm in summary["arms"]:
    b = u["by_arm"][arm]
    medyan = (b["cost_usd_per_run"] or {}).get("median") if b["cost_usd_per_run"] else None
    print(f"{arm:28s} {b['runs']:6d} {b['successful_calls']:7d} "
          f"{f(b['input_tokens'], '10d')} {f(b['output_tokens'], '10d')} "
          f"{f(b['cost_usd'], '10.4f')} {f(medyan, '14.6f')}")

o = u["infrastructure_overhead"]
print(f"\naltyapı overhead (ölçüm üretmeyen koşular): {o['runs']} koşu, "
      f"${o['cost_usd'] if o['cost_usd'] is not None else '—'}")
print(f"sağlayıcı hatası denemesi: {u['scored_runs']['provider_error_attempts']}")

## Tablo 4 — Sözleşme uyumu (RQ2/RQ3)

Birim **arm-run**'dır, görev değil: bunlar süreç betimleyicileridir, birincil estimand değil.
`contract` grafında ikinci denemeye yalnız doğrulama başarısızsa geçilir; bu yüzden
`attempt_count > 1` ilk denemenin uyumsuz olduğunu zaten kanıtlar.

In [ ]:
for arm, c in summary["compliance"].items():
    print(f"\n[{arm}]  (birim: {c['unit']}, {c['runs']} koşu)")
    print(f"  parse başarısı            {c['parse_ok_count']}/{c['runs']} = {c['parse_ok_rate']}")
    if "final_compliance" not in c:
        continue  # structured'da doğrulama kapısı yok
    print(f"  ilk deneme uyumu          {c['first_attempt_valid_count']}/{c['runs']} "
          f"= {c['first_attempt_compliance']}")
    print(f"  nihai uyum                {c['final_valid_count']}/{c['runs']} "
          f"= {c['final_compliance']}")
    print(f"  ortalama deneme sayısı    {c['mean_attempt_count']}  "
          f"(dağılım: {c['attempt_count_distribution']}, tavan {c['max_planner_attempts']})")
    print(f"  retry ile kurtarma        {c['retry_recovered_count']}/{c['retried_count']} "
          f"= {c['retry_recovery_rate']}   ← payda: retry'a fiilen girenler")
    print(f"  validation exhaustion     {c['validation_exhausted_count']}/{c['runs']} "
          f"= {c['validation_exhaustion_rate']}")

## Görev bazında dağılım — hangi görevlerde etki var?

In [ ]:
a, b = summary["primary_comparison"].split("_vs_")
rates = {(r["task_id"], r["arm"]): float(r["plus_rate"]) for r in task_level}
tasks = sorted({r["task_id"] for r in task_level})
for task_id in sorted(tasks, key=lambda t: rates[(t, a)] - rates[(t, b)]):
    diff = rates[(task_id, a)] - rates[(task_id, b)]
    if diff:
        print(f"{task_id:24s} {a}={rates[(task_id, a)]:.2f}  "
              f"{b}={rates[(task_id, b)]:.2f}  fark {diff:+.2f}")